# Install

In [ ]:
!pip3 install pshmodule

In [ ]:
!pip3 install pickle5

In [ ]:
!pip3 install pandas==1.5.0

In [ ]:
!pip3 install swifter

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Load

In [76]:
import sys
sys.path.append('/content/drive/MyDrive/Advertising_By_Personality/src/preprocessing')
print(sys.path)

['/content', '/env/python', '/usr/lib/python39.zip', '/usr/lib/python3.9', '/usr/lib/python3.9/lib-dynload', '', '/usr/local/lib/python3.9/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.9/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/Advertising_By_Personality/src/preprocessing', '/content/drive/MyDrive/Advertising_By_Personality/src/preprocessing']


In [84]:
import itertools
import swifter
import config as cfg
import pandas as pd
from tqdm import tqdm
from pshmodule.utils import filemanager as fm

In [78]:
df = fm.load(cfg.origin_data)

extension : .xlsx
Loaded 25000 records from /content/drive/MyDrive/Advertising_By_Personality/data/origin/origin.xlsx


In [79]:
df.rename(columns={'관리번호':'no', '광고 구분':'advertisement_type', '마케팅전략':'marketing_strategy', '일상정보':'daily_information', '시즌정보':'season_information', '분류':'type', '제목':'title', '본문':'content', '납품차수':'degree'}, inplace=True)

In [80]:
df.head()

,no,advertisement_type,marketing_strategy,daily_information,season_information,type,title,content,degree
0,1,NaN,NaN,NaN,NaN,원문,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰,"마지막 2일! [계절밥상 디너/주말 1만원, 런치 5천원 할인 쿠폰] 9월30일까지...",NaN
1,1,할인/쿠폰/혜택,"매력적인숫자(시간,돈)",NaN,NaN,ST,★최대 1만원★ 계절밥상 할인쿠폰,"이.틀.만! VIP 고객 디너&주말 1만원, 런치 5천원 할인 쿠폰 놓치지 마세요!",1차
2,1,할인/쿠폰/혜택,핵심정보만 전달,NaN,NaN,SF,ONLY VIP! 할인쿠폰♬,소중한 이들과 더 든든하게 먹고 싶다면?\\[계절밥상] 디너/주말 1만원+런치 5천...,1차
3,1,할인/쿠폰/혜택,호기심유발(질문),NaN,NaN,NT,이 혜택💰 놓치지 않을 거예요,"[계절밥상] 디너/주말 1O,OOO원, 런치 5,OOO원 할인\\9월 30일까지🚨",1차
4,1,할인/쿠폰/혜택,호기심유발(이모지),NaN,NaN,NF,VIP님들만을 위한 쿠폰 도착💌,계절밥상에서 든든한 한끼🍱 어때요?\\런치/디너/주말 할인 쿠폰(~9/30) 사용하...,1차


# Processing

In [81]:
df_temp = df.copy()

In [82]:
df_temp = df_temp[['no', 'advertisement_type', 'marketing_strategy', 'daily_information', 'season_information', 'type', 'title', 'content', 'degree']]

In [85]:
result = []
temp = []

for i in df_temp.iterrows():
  temp.append([i[1]['type'], i[1]['title'] + '\\\\'  + i[1]['content'], i[1]['season_information']])
  if len(temp) == 5:
    index_combinations = list(itertools.combinations(temp, 2))
    for c in index_combinations:
      # c[0][0] 바뀌기 전, c[1][0] 바뀐 후
      result.append([c[0][0], c[1][0], c[0][1], c[1][1], c[1][2]])
    for c in index_combinations:
      result.append([c[1][0], c[0][0], c[1][1], c[0][1], c[0][2]])
    temp = []

In [86]:
len(result)

100000

In [87]:
result[45000]

['원문',
 'ST',
 '새해부터 깊카가 챙겨주는 페이백과 경품!\\\\기프트카드를 구매하면 선착순 3천원 페이백 깊카와 구찌,프라다,루이비통 카드지갑까지\\uD83C\\uDFB0',
 '받으면서 시작하는 한 해♥\\\\기프트카드 구매만 해도 ✔️선착순 3천원 페이백 ✔️구찌, 프라다, 루이비틍 명품 카드지갑 증정\\\\새해된 기념으로 쏜다!',
 '새해']

In [88]:
df_result = pd.DataFrame(result, columns=['ctrl1', 'ctrl2', 'input', 'label', 'season'])

# shuffle & extract 10 

In [95]:
df_null = df_result[df_result.season.isnull()]
df_notnull = df_result[df_result.season.notnull()]

In [96]:
df_temp_null = df_null.sample(frac=1).reset_index(drop=True)
df_temp_notnull = df_notnull.sample(frac=1).reset_index(drop=True)

In [97]:
df_temp_null = df_temp_null.iloc[:5]
df_temp_notnull = df_temp_notnull.iloc[:5]

In [98]:
df_temp_null.head()

,ctrl1,ctrl2,input,label,season
0,원문,NT,화제의 갤럭시 노트10\ud83d\udcf1\\댓글만 써도 100% 당첨+온라인체험존 ▶,핸드폰 댓글로 장만했어요😎\\갤럭시 노트10 댓.글.만.달.고 가져가자🎁\\참여만 ...,NaN
1,ST,NF,"굶기만 하는 다이어트는 이제 그만❌\\건강하고 맛있는 다이어트 간식 GET하고 3,...",프로 다이어터들 다 모이세요🏃\\이 기회 혼자만 놓치면 땅을 치고 후회할걸요?!😭 ...,NaN
2,원문,NF,사용하신 포인트 돌려드립니다.\\오늘 마감! 페이백 신청하러 가기▶,왜 내 포인트는 마르지 않는 고양😻\\두근두근❤️ 페이백 신청했더니 사용한 포인트가...,NaN
3,NF,원문,아니 진짜 이래도 되는거야?!🔥\\모르고 있었던 분들은 인생의 절반 손해본 겁니다😎...,CJmall 어메이징 세일위크 2탄 오픈!\\매일 최저가 도전! 해피한 특가! 지금...,NaN
4,ST,NF,꽝 없는 당첨 기회 도착★\\주사위 던지고 발뮤다 공기청정기 득템♬\\지금 바로 행...,행운의 주사위 굴릴 찬스🎲\\고객님께만 드릴거라눙💘 발뮤다 공기청정기 행운이 뙇- ...,NaN


In [99]:
df_temp_notnull.head()

,ctrl1,ctrl2,input,label,season
0,ST,NT,고객님의 신년 운세 도착!\\CJmall앱에서 공짜로 운세 확인✔️\\행운 선물도 ...,내년 나의 운세는?🔮\\CJmall에서 무/료/로 새해 타로도 보고 푸짐한 경품도 ...,새해
1,원문,SF,아직 구매하시지 않으셨나요?\\방문만 해도 기프티콘 100% 증정,"올겨울 SNS대란템 고객님만 없네?!\\날 추울 땐, 침대 위에서 랜선 쇼핑하고 마...",겨울맞이
2,ST,SF,설날은 [더플레이스 쿠폰]♬\\다운받은 쿠폰 확인하고 더플레이스 가자!,설연휴엔 더플레이스와 함께~\\모던한 분위기의 가족 모임 장소로 PICK❗\\요즘 ...,설연휴
3,원문,NT,CJmall 설맞이 혜택받쥐\\2020년 무료 신년운세 + 식품 10% 적립,복채는 필요하지 않아요🙅‍♀️\\[CJmall]에서 설을 맞아 무.료.로 신년 운세...,설연휴
4,원문,NF,\ud83c\udf84크리스마스 선물 쏩니다!\ud83c\udf84\\2021년 운...,내년에는 남친 생기게 해주세요🙏\\크리스마스 선물로 신년운세에서 알려준다는데?💡\\...,크리스마스


In [100]:
df = pd.concat([df_temp_null, df_temp_notnull])

In [101]:
df.head(10)

,ctrl1,ctrl2,input,label,season
0,원문,NT,화제의 갤럭시 노트10\ud83d\udcf1\\댓글만 써도 100% 당첨+온라인체험존 ▶,핸드폰 댓글로 장만했어요😎\\갤럭시 노트10 댓.글.만.달.고 가져가자🎁\\참여만 ...,NaN
1,ST,NF,"굶기만 하는 다이어트는 이제 그만❌\\건강하고 맛있는 다이어트 간식 GET하고 3,...",프로 다이어터들 다 모이세요🏃\\이 기회 혼자만 놓치면 땅을 치고 후회할걸요?!😭 ...,NaN
2,원문,NF,사용하신 포인트 돌려드립니다.\\오늘 마감! 페이백 신청하러 가기▶,왜 내 포인트는 마르지 않는 고양😻\\두근두근❤️ 페이백 신청했더니 사용한 포인트가...,NaN
3,NF,원문,아니 진짜 이래도 되는거야?!🔥\\모르고 있었던 분들은 인생의 절반 손해본 겁니다😎...,CJmall 어메이징 세일위크 2탄 오픈!\\매일 최저가 도전! 해피한 특가! 지금...,NaN
4,ST,NF,꽝 없는 당첨 기회 도착★\\주사위 던지고 발뮤다 공기청정기 득템♬\\지금 바로 행...,행운의 주사위 굴릴 찬스🎲\\고객님께만 드릴거라눙💘 발뮤다 공기청정기 행운이 뙇- ...,NaN
0,ST,NT,고객님의 신년 운세 도착!\\CJmall앱에서 공짜로 운세 확인✔️\\행운 선물도 ...,내년 나의 운세는?🔮\\CJmall에서 무/료/로 새해 타로도 보고 푸짐한 경품도 ...,새해
1,원문,SF,아직 구매하시지 않으셨나요?\\방문만 해도 기프티콘 100% 증정,"올겨울 SNS대란템 고객님만 없네?!\\날 추울 땐, 침대 위에서 랜선 쇼핑하고 마...",겨울맞이
2,ST,SF,설날은 [더플레이스 쿠폰]♬\\다운받은 쿠폰 확인하고 더플레이스 가자!,설연휴엔 더플레이스와 함께~\\모던한 분위기의 가족 모임 장소로 PICK❗\\요즘 ...,설연휴
3,원문,NT,CJmall 설맞이 혜택받쥐\\2020년 무료 신년운세 + 식품 10% 적립,복채는 필요하지 않아요🙅‍♀️\\[CJmall]에서 설을 맞아 무.료.로 신년 운세...,설연휴
4,원문,NF,\ud83c\udf84크리스마스 선물 쏩니다!\ud83c\udf84\\2021년 운...,내년에는 남친 생기게 해주세요🙏\\크리스마스 선물로 신년운세에서 알려준다는데?💡\\...,크리스마스


In [102]:
fm.save('/content/drive/MyDrive/Advertising_By_Personality/data/origin/predict_check.xlsx', df)

Saved 10 records
